# Rung 3 — QAT (102M tokens)

Absmean g=128 scales, STE, frozen BF16 teacher KD, resumed from the best PTQ checkpoint. Resumable because a Colab session expires mid-run.

## Setup

In [ ]:
# Colab Pro A100 setup. Persist results/checkpoints on Drive so a session
# expiry costs the run, not the artifacts.
from google.colab import drive
drive.mount("/content/drive")
REPO = "/content/drive/MyDrive/lfm3-ternary"
import os, subprocess, sys
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/ORG/LFM3-Ternary", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
os.makedirs("results", exist_ok=True)
!pip -q install "transformers>=5.12.1" "lm-eval==0.4.7" accelerate datasets sentencepiece safetensors gguf
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


## Training corpus

In [ ]:
!python3 scripts/build_qat_data.py --model LiquidAI/LFM3-1.7B \
    --out-dir data/calib/qat --tokens 102000000

## Train (resume by re-running this cell)

In [ ]:
!python3 scripts/train_qat.py --model checkpoints/rung2/hf --teacher LiquidAI/LFM3-1.7B \
    --data data/calib/qat/corpus.pt --out checkpoints/rung3 --tokens 102000000 \
    --save-every 200
# to resume after an expiry, add: --resume checkpoints/rung3/stepN.pt

## Gate

In [ ]:
!bash scripts/eval_lm_eval.sh checkpoints/rung3/hf results/rung3.json
!python3 scripts/gates.py tier0 --baseline results/baseline.json --scores results/rung3.json \
    --rung 3 --remaining 0 --out results/rung3_verdict.json